In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    accuracy_score
)

In [2]:

def load_and_prepare_data():
    """Load standard dataset and convert to pandas DataFrame."""
    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['target'] = data.target
    print(f"Dataset Shape: {df.shape}")
    print(f"Target Distribution:\n{df['target'].value_counts()}\n")
    return df, data.target_names

In [3]:
def preprocess_data(df):
    """Separate features and target, perform train-test split and scaling."""
    X = df.drop(columns=['target'])
    y = df['target']
    
    # Train-Test Split (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    
    # Feature Scaling (Standardisation)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test, X.columns, scaler

In [4]:
def train_model(X_train, y_train):
    """Train Random Forest Classifier with Grid Search CV."""
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }
    
    rf = RandomForestClassifier(random_state=42)
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=5,
        scoring='roc_auc',
        n_jobs=-1
    )
    
    grid_search.fit(X_train, y_train)
    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Best CV ROC-AUC Score: {grid_search.best_score_:.4f}\n")
    
    return grid_search.best_estimator_

In [5]:
def evaluate_model(model, X_test, y_test, target_names):
    """Evaluate model performance on unseen test data."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print("--- Classification Report ---")
    print(classification_report(y_test, y_pred, target_names=target_names))
    
    acc = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    print(f"Test Accuracy:  {acc * 100:.2f}%")
    print(f"Test ROC-AUC:    {roc_auc:.4f}\n")
    
    # Confusion Matrix Plot
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    plt.close()
    print("Saved 'confusion_matrix.png' successfully.")

In [6]:
def plot_feature_importance(model, feature_names, top_n=10):
    """Plot top N feature importances."""
    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1][:top_n]
    
    plt.figure(figsize=(10, 6))
    plt.title(f'Top {top_n} Feature Importances')
    plt.bar(range(top_n), importances[indices], align='center')
    plt.xticks(range(top_n), [feature_names[i] for i in indices], rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('feature_importance.png')
    plt.close()
    print("Saved 'feature_importance.png' successfully.")

In [7]:
if __name__ == "__main__":
    print("=== Starting Real ML Pipeline ===\n")
    
    # 1. Load Data
    df, target_names = load_and_prepare_data()
    
    # 2. Preprocess & Split
    X_train, X_test, y_train, y_test, feature_names, scaler = preprocess_data(df)
    
    # 3. Train Model with Hyperparameter Tuning
    best_model = train_model(X_train, y_train)
    
    # 4. Evaluate Model
    evaluate_model(best_model, X_test, y_test, target_names)
    
    # 5. Feature Importances
    plot_feature_importance(best_model, feature_names)
    
    print("\n=== Pipeline Execution Completed Successfully ===")

=== Starting Real ML Pipeline ===

Dataset Shape: (569, 31)
Target Distribution:
target
1    357
0    212
Name: count, dtype: int64

Best Parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}
Best CV ROC-AUC Score: 0.9875

--- Classification Report ---
              precision    recall  f1-score   support

   malignant       0.95      0.93      0.94        42
      benign       0.96      0.97      0.97        72

    accuracy                           0.96       114
   macro avg       0.96      0.95      0.95       114
weighted avg       0.96      0.96      0.96       114

Test Accuracy:  95.61%
Test ROC-AUC:    0.9939

Saved 'confusion_matrix.png' successfully.
Saved 'feature_importance.png' successfully.

=== Pipeline Execution Completed Successfully ===
